In [5]:
# Reusable diarization module
# (This notebook now uses `src/diarize.py` so the pipeline can be imported elsewhere.)

import os
import sys
import getpass
from pathlib import Path

# Ensure API key is available (notebook-friendly prompt)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ").strip()

# Make sure we can import from src/ whether the notebook is run from repo root or src/
CWD = Path.cwd()
if (CWD / "src").is_dir():
    sys.path.insert(0, str(CWD / "src"))
elif CWD.name == "src":
    sys.path.insert(0, str(CWD))

# Example: diarize + transcribe a local audio file
# (Adjust the path to your own uploaded/recorded file.)
# audio_path = (CWD / "src" / "tim.wav") if (CWD / "src").is_dir() else (CWD / "tim.wav")
# result = diarizer.diarize_audio(str(audio_path))

# result.text[:400], result.segments[:3]


In [6]:
import base64
import os
import getpass
import shutil
import subprocess
from typing import Any, Iterable
from openai import OpenAI

In [7]:
# Prefer environment variable, but fall back to a secure prompt in notebooks.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ").strip()

client = OpenAI()

In [8]:
def _seg_get(seg: Any, key: str):
    """Support both OpenAI segment objects and dicts."""
    if hasattr(seg, key):
        return getattr(seg, key)
    return seg[key]


def merge_same_speaker_segments(
    segments: Iterable[Any],
    *,
    max_gap_s: float = 0.25,
):
    """Merge adjacent segments with the same speaker.

    This turns many short diarization segments into longer continuous stretches.
    """
    segs = sorted(segments, key=lambda s: float(_seg_get(s, "start")))
    merged: list[dict[str, Any]] = []

    for s in segs:
        speaker = _seg_get(s, "speaker")
        start = float(_seg_get(s, "start"))
        end = float(_seg_get(s, "end"))

        if not merged:
            merged.append({"speaker": speaker, "start": start, "end": end})
            continue

        prev = merged[-1]
        if speaker == prev["speaker"] and start <= prev["end"] + max_gap_s:
            prev["end"] = max(prev["end"], end)
        else:
            merged.append({"speaker": speaker, "start": start, "end": end})

    return merged


def find_solo_reference_clips(
    segments: Iterable[Any],
    *,
    speaker: str,
    min_duration_s: float = 2.0,
    max_duration_s: float = 10.0,
    merge_gap_s: float = 0.25,
    edge_padding_s: float = 0.15,
    overlap_tolerance_s: float = 0.0,
    take: str = "center",  # "start" | "center"
):
    """Find 2–10s windows where `speaker` talks uninterrupted.

    Returns a list of dicts like {start_s, duration_s, solo_span_s}.

    Notes:
    - "Solo" here means: within the proposed clip window (plus optional edge padding),
      there are no segments attributed to a different speaker.
    - If a merged solo span is longer than `max_duration_s`, we select a max-length
      window from inside it (start-anchored or centered).
    """
    if min_duration_s < 0 or max_duration_s <= 0 or min_duration_s > max_duration_s:
        raise ValueError("Invalid min/max durations")

    merged = merge_same_speaker_segments(segments, max_gap_s=merge_gap_s)

    # For overlap checks, keep the raw segments.
    raw = list(segments)

    def has_other_speaker_overlap(start_s: float, end_s: float) -> bool:
        check_start = start_s - edge_padding_s
        check_end = end_s + edge_padding_s
        for seg in raw:
            spk = _seg_get(seg, "speaker")
            if spk == speaker:
                continue
            s0 = float(_seg_get(seg, "start"))
            s1 = float(_seg_get(seg, "end"))
            overlap = max(0.0, min(check_end, s1) - max(check_start, s0))
            if overlap > overlap_tolerance_s:
                return True
        return False

    clips: list[dict[str, float]] = []
    for span in merged:
        if span["speaker"] != speaker:
            continue

        span_start = float(span["start"])
        span_end = float(span["end"])
        span_dur = span_end - span_start
        if span_dur < min_duration_s:
            continue

        clip_dur = min(max_duration_s, span_dur)
        if take == "center" and span_dur > clip_dur:
            clip_start = span_start + (span_dur - clip_dur) / 2.0
        else:
            clip_start = span_start

        clip_end = clip_start + clip_dur
        if not has_other_speaker_overlap(clip_start, clip_end):
            clips.append(
                {
                    "start_s": float(clip_start),
                    "duration_s": float(clip_dur),
                    "solo_span_s": float(span_dur),
                }
            )

    # Prefer longer uninterrupted spans, then longer clip durations.
    clips.sort(key=lambda c: (c["solo_span_s"], c["duration_s"]), reverse=True)
    return clips


def pick_best_solo_clip_by_speaker(
    segments: Iterable[Any],
    speakers: list[str],
    *,
    min_duration_s: float = 2.0,
    max_duration_s: float = 10.0,
):
    """Convenience: return the best solo clip per speaker (if found)."""
    best: dict[str, dict[str, float]] = {}
    for spk in speakers:
        clips = find_solo_reference_clips(
            segments,
            speaker=spk,
            min_duration_s=min_duration_s,
            max_duration_s=max_duration_s,
        )
        if clips:
            best[spk] = clips[0]
    return best

In [9]:
def to_data_url(path: str) -> str:
    """Encode an audio file as a data URL for known_speaker_references[]."""
    ext = os.path.splitext(path)[1].lower()
    mime_by_ext = {
        ".wav": "audio/wav",
        ".mp3": "audio/mpeg",
        ".m4a": "audio/mp4",
        ".mp4": "audio/mp4",
        ".ogg": "audio/ogg",
        ".flac": "audio/flac",
        ".webm": "audio/webm",
    }
    mime = mime_by_ext.get(ext, "application/octet-stream")

    with open(path, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"


def slice_reference_clip(
    *,
    input_path: str,
    output_wav_path: str,
    start_s: float,
    duration_s: float,
    sample_rate_hz: int = 16000,
) -> None:
    """Create a clean 2–10s mono WAV reference clip using ffmpeg.

    The API requires each known speaker reference sample to be between 2 and 10 seconds.

    Args:
        input_path: Path to the input audio file.
        output_wav_path: Path to save the output WAV file.
        start_s: Start time in seconds.
        duration_s: Duration in seconds.
        sample_rate_hz: Sample rate in Hz.
    """
    if not (2.0 <= duration_s <= 10.0):
        raise ValueError("Reference duration must be between 2 and 10 seconds")
    if shutil.which("ffmpeg") is None:
        raise RuntimeError(
            "ffmpeg not found. Install it (e.g. `brew install ffmpeg`) or provide an existing 2–10s WAV clip."
        )

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-ss",
            str(start_s),
            "-t",
            str(duration_s),
            "-i",
            input_path,
            "-ac",
            "1",
            "-ar",
            str(sample_rate_hz),
            output_wav_path,
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

In [10]:
def diarize_audio(
    path: str,
    *,
    known_speaker_names: list[str] | None = None,
    known_speaker_references: list[str] | None = None,
    language: str | None = "en",
    chunking_strategy: str | dict = "auto",
    timeout_s: float | None = 300.0,
    verbose_segments: bool = False,
):
    """Diarize + transcribe audio.

    Notes:
    - If you only have 2 real people but see A/B/C labels, that's usually diarization
      over-splitting (overlap, noise, short backchannels). You can post-process to 2.
    - `known_speaker_names/references` help, but the model can still emit extra labels
      for audio that doesn't match either reference well.
    """
    with open(path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="gpt-4o-transcribe-diarize",
            file=audio_file,
            response_format="diarized_json",
            chunking_strategy=chunking_strategy,
            language=language,
            known_speaker_names=known_speaker_names,
            known_speaker_references=known_speaker_references,
            timeout=timeout_s,
        )

    if verbose_segments:
        for segment in transcript.segments:
            print(segment.speaker, segment.text, segment.start, segment.end)

    return transcript.text, transcript.segments

In [11]:
def relabel_target_vs_other(segments, *, target_name: str, other_label: str = "other"):
    """Collapse diarization labels into {target_name, other_label}."""
    relabeled = []
    for s in segments:
        speaker = _seg_get(s, "speaker")
        relabeled.append(
            {
                "speaker": target_name if speaker == target_name else other_label,
                "start": float(_seg_get(s, "start")),
                "end": float(_seg_get(s, "end")),
                "text": _seg_get(s, "text"),
            }
        )
    return relabeled


def auto_other_reference_clip(
    segments,
    *,
    target_name: str,
    input_audio_path: str,
    output_wav_path: str = "other_ref.wav",
    duration_s: float = 9.0,
    merge_gap_s: float = 0.25,
):
    """Pick a long non-target solo span and cut it as an "other" reference.

    This is useful when you only have a stored reference for the target person, but
    you want to *also* anchor "other" to reduce label drift and extra speakers.
    """
    merged = merge_same_speaker_segments(segments, max_gap_s=merge_gap_s)

    # Find the longest merged span that is NOT the target.
    best = None
    for span in merged:
        if span["speaker"] == target_name:
            continue
        span_dur = float(span["end"] - span["start"])
        if best is None or span_dur > best["solo_span_s"]:
            best = {
                "start_s": float(span["start"]),
                "solo_span_s": float(span_dur),
            }

    if best is None:
        raise RuntimeError("Could not find any non-target span to use as an other reference")

    # Take a 2–10s clip from inside the span (centered if span is longer).
    clip_dur = max(2.0, min(10.0, float(duration_s), best["solo_span_s"]))
    if best["solo_span_s"] > clip_dur:
        clip_start = best["start_s"] + (best["solo_span_s"] - clip_dur) / 2.0
    else:
        clip_start = best["start_s"]

    slice_reference_clip(
        input_path=input_audio_path,
        output_wav_path=output_wav_path,
        start_s=clip_start,
        duration_s=clip_dur,
    )

    return {"path": output_wav_path, "start_s": clip_start, "duration_s": clip_dur}


def diarize_target_vs_other(
    audio_path: str,
    *,
    target_name: str,
    target_reference_wav: str,
    other_label: str = "other",
    two_pass: bool = True,
    auto_build_other_reference: bool = True,
    other_reference_wav: str = "other_ref.wav",
    language: str = "en",
    verbose_steps: bool = True,
    verbose_segments: bool = True,
    chunking_strategy: str | dict = "auto",
):
    """
    Diarize with one stored target identity, label everyone else as `other_label`.

    Process:
    - Pass the target reference first. The model will label segments for that
      person as target_name, and everyone else as A, B, etc.
    - If two_pass is True, cut an "other" reference clip from the non-target stretches.
    - Re-run with both references: [target_name, other_label].
    - Return normalized segments where speaker is target_name or other_label.
    """

    if verbose_steps:
        print("Pass 1/{}: anchoring target".format(2 if two_pass else 1))

    # Pass 1: anchor the target.
        text1, segs1 = diarize_audio(
            audio_path,
            known_speaker_names=[target_name],
            known_speaker_references=[to_data_url(target_reference_wav)],
            language=language,
            chunking_strategy=chunking_strategy,
        )

    if not two_pass:
        # Collapse all non-target labels (A/B/...) to `other_label`.
        return text1, relabel_target_vs_other(segs1, target_name=target_name, other_label=other_label)

    if verbose_steps:
        print("Pass 2/2: anchoring other")

    if auto_build_other_reference:
        other_meta = auto_other_reference_clip(
            segs1,
            target_name=target_name,
            input_audio_path=audio_path,
            output_wav_path=other_reference_wav,
        )
        other_ref = other_meta["path"]
    else:
        other_ref = other_reference_wav
 
    # Pass 2: anchor both sides.
    text2, segs2 = diarize_audio(
        audio_path,
        known_speaker_names=[target_name, other_label],
        known_speaker_references=[to_data_url(target_reference_wav), to_data_url(other_ref)],
        language=language,
        chunking_strategy=chunking_strategy,
    )

    return text2, relabel_target_vs_other(segs2, target_name=target_name, other_label=other_label)

In [20]:
TARGET_NAME = "michelle"
TARGET_REF = "/Users/timleong/guidepost/test/michelle.m4a"
audio_path = "/Users/timleong/guidepost/datasets/self_recorded_data/scenario_1_michelle_1.m4a"

text, segments_norm = diarize_target_vs_other(
    audio_path,
    target_name=TARGET_NAME,
    target_reference_wav=TARGET_REF,
    auto_build_other_reference=True,
    other_reference_wav="other_ref.wav",
    two_pass=False
)

Pass 1/1: anchoring target


In [21]:
segments_norm

[{'speaker': 'michelle',
  'start': 0.30000000000000004,
  'end': 0.9000000000000001,
  'text': ' Who would you,'},
 {'speaker': 'michelle',
  'start': 1.0000000000000002,
  'end': 2.3500000000000005,
  'text': ' doing'},
 {'speaker': 'other',
  'start': 2.3500000000000005,
  'end': 4.3999999999999995,
  'text': ' Hey, Michelle doing good.'},
 {'speaker': 'other',
  'start': 5.449999999999999,
  'end': 6.049999999999999,
  'text': ' Yeah,'},
 {'speaker': 'other',
  'start': 6.149999999999999,
  'end': 7.749999999999999,
  'text': ' I feel like things are just doing all right.'},
 {'speaker': 'michelle',
  'start': 9.1,
  'end': 9.549999999999999,
  'text': ' all right?'},
 {'speaker': 'michelle',
  'start': 10.249999999999998,
  'end': 14.999999999999996,
  'text': ' Any noteworthy stuff in your projects that have been coming up?'},
 {'speaker': 'other',
  'start': 18.049999999999997,
  'end': 18.549999999999997,
  'text': ' No,'},
 {'speaker': 'other',
  'start': 18.65,
  'end': 22.45

In [22]:
from pathlib import Path
import json

out_path = Path("test") / "segments_norm.json"
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("w", encoding="utf-8") as f:
    json.dump(segments_norm, f, indent=2, ensure_ascii=False, default=str)

print(f"Wrote {len(segments_norm)} segments to {out_path.resolve()}")


Wrote 88 segments to /Users/timleong/guidepost/test/test/segments_norm.json


In [23]:
TARGET_NAME = "tim"
TARGET_REF = "/Users/timleong/guidepost/test/tim.wav"
audio_path = "/Users/timleong/guidepost/datasets/self_recorded_data/tim_scenario_3_1.m4a"

text, segments_norm = diarize_target_vs_other(
    audio_path,
    target_name=TARGET_NAME,
    target_reference_wav=TARGET_REF,
    auto_build_other_reference=True,
    other_reference_wav="other_ref.wav",
    two_pass=False
)

segments_norm

Pass 1/1: anchoring target


[{'speaker': 'other',
  'start': 2.196,
  'end': 3.7460000000000004,
  'text': " Hey killing hey, how's it going?"},
 {'speaker': 'other',
  'start': 4.096,
  'end': 5.446,
  'text': ' Hey, doing alright.'},
 {'speaker': 'other', 'start': 7.124, 'end': 7.524, 'text': ' Cool.'},
 {'speaker': 'other',
  'start': 8.274,
  'end': 11.823999999999998,
  'text': ' So yeah, just to level set, this is our quarterly check in.'},
 {'speaker': 'other',
  'start': 12.373999999999999,
  'end': 17.774,
  'text': ' What I want to kind of go over is just how you think this past quarter went.'},
 {'speaker': 'other',
  'start': 18.374,
  'end': 21.323999999999998,
  'text': ' I kind of think about some of the successes you had,'},
 {'speaker': 'other',
  'start': 21.474,
  'end': 28.123999999999995,
  'text': ' some of the areas where you struggled and just kind of generally talk about how you feel about your career growth here,'},
 {'speaker': 'other',
  'start': 28.273999999999994,
  'end': 33.1739999

In [ ]:
segments_norm.to_json()

[{'speaker': 'other',
  'start': 2.196,
  'end': 3.396,
  'text': ' Hey Killian. Hey,'},
 {'speaker': 'other',
  'start': 3.4459999999999997,
  'end': 3.7459999999999996,
  'text': " how's it going?"},
 {'speaker': 'other',
  'start': 4.045999999999999,
  'end': 5.645999999999999,
  'text': ' Hey, doing all right.'},
 {'speaker': 'other',
  'start': 7.124,
  'end': 7.6739999999999995,
  'text': ' Cool.'},
 {'speaker': 'other',
  'start': 8.274,
  'end': 10.024,
  'text': ' So yeah, just to level set,'},
 {'speaker': 'other',
  'start': 10.123999999999999,
  'end': 11.774,
  'text': ' this is our quarterly check-in.'},
 {'speaker': 'other',
  'start': 12.424,
  'end': 17.724,
  'text': ' What I want to kind of go over is just how you think this past quarter went.'},
 {'speaker': 'other',
  'start': 18.424,
  'end': 21.324,
  'text': ' I kind of think about some of the successes you had,'},
 {'speaker': 'other',
  'start': 21.524,
  'end': 28.074,
  'text': ' some of the areas where you 